# 10 - YOLO Dataset Preparation

**Phase 3, Step 3.2 - Dataset Preparation for Detection**

Reuses the raw collection from `01_Dataset_Loading.ipynb` (full images + all matching boxes) and converts it into YOLO format: normalized `class x_center y_center width height` label files, an images/labels train/val/test split, and a `data.yaml` config for Ultralytics YOLOv8.

### 1. Mount Drive and load configuration

In [1]:
!pip install -q datasets pyyaml tqdm scikit-learn matplotlib pandas opencv-python-headless pyyaml pillow

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

# ---------------------------------------------------------------------------
# Project configuration - shared across every SmartVision AI notebook.
# All notebooks read/write under this same Google Drive folder so that
# work done in one notebook (e.g. dataset collection) is available to the
# next one (e.g. training), even across separate Colab sessions.
# ---------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/SmartVisionAI"

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
RAW_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "images")
RAW_ANNOTATIONS_PATH = os.path.join(RAW_DATA_DIR, "annotations.json")

CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")
CLASSIFICATION_TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
CLASSIFICATION_VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
CLASSIFICATION_TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

DETECTION_DIR = os.path.join(BASE_DIR, "detection")
DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")
DETECTION_YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for d in [BASE_DIR, RAW_DATA_DIR, RAW_IMAGES_DIR, CLASSIFICATION_DIR, DETECTION_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# The 25 selected COCO classes (must match COCO category names exactly)
SELECTED_CLASSES = [
    # Vehicles (6)
    "car", "truck", "bus", "motorcycle", "bicycle", "airplane",
    # Person (1)
    "person",
    # Outdoor (3)
    "traffic light", "stop sign", "bench",
    # Animals (6)
    "dog", "cat", "horse", "bird", "cow", "elephant",
    # Kitchen & food (5)
    "bottle", "cup", "bowl", "pizza", "cake",
    # Furniture & indoor (4)
    "chair", "couch", "bed", "potted plant",
]
assert len(SELECTED_CLASSES) == 25

CLASS_TO_IDX = {name: i for i, name in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(SELECTED_CLASSES)}

def safe_name(class_name):
    return class_name.replace(" ", "_")

IMAGES_PER_CLASS = 350        # -> 8,750 images total (up from 100/class to fight overfitting)
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

CLS_IMG_SIZE = 224            # Classification input resolution (single-resolution throughout)
FINE_TUNE_IMG_SIZE = 384      # Unused by classifier training (reverted to single-resolution); kept for compatibility
YOLO_IMG_SIZE = 640
BATCH_SIZE = 32                # Stage 1 batch size
BATCH_SIZE_STAGE2 = 16         # Smaller batch at 384x384 to fit GPU memory (~2.9x pixels/image)

HF_DATASET_NAME = "detection-datasets/coco"

print("BASE_DIR:", BASE_DIR)
print("Classes:", len(SELECTED_CLASSES))


BASE_DIR: /content/drive/MyDrive/SmartVisionAI
Classes: 25


In [4]:
import json
import random
import shutil
import yaml

random.seed(42)

with open(RAW_ANNOTATIONS_PATH) as f:
    annotations = json.load(f)
print(f"Loaded {len(annotations)} annotated images.")


Loaded 3087 annotated images.


### 2. Convert corner-format boxes to normalized YOLO format

**Confirmed bbox format:** `detection-datasets/coco` stores `bbox` as `[x_min, y_min, x_max, y_max]` (corner format), verified empirically - 100% of boxes are valid under this interpretation vs. only ~24% under the originally assumed `[x, y, width, height]` format. Getting this wrong is what caused YOLOv8 to discard the majority of images as "corrupt" (out-of-bounds normalized coordinates) in earlier runs.

In [5]:
import shutil as _shutil
# Clear any stale detection data from a previous run before rebuilding.
# Without this, re-running this notebook after 01_Dataset_Loading collected a
# different number of images would leave old files mixed in with new ones -
# including old files from before any bbox-format fix, silently corrupting results.
_shutil.rmtree(DETECTION_IMAGES_DIR, ignore_errors=True)
_shutil.rmtree(DETECTION_LABELS_DIR, ignore_errors=True)
os.makedirs(DETECTION_IMAGES_DIR, exist_ok=True)
os.makedirs(DETECTION_LABELS_DIR, exist_ok=True)

yolo_records = []  # (image_filename, list_of_label_lines)
skipped_degenerate = 0

for ann in annotations:
    w, h = ann["width"], ann["height"]
    label_lines = []
    for box in ann["boxes"]:
        x1, y1, x2, y2 = box["bbox"]  # x_min, y_min, x_max, y_max
        # Defensive clamp in case of minor annotation overshoot at image edges
        x1 = max(0.0, min(x1, w))
        y1 = max(0.0, min(y1, h))
        x2 = max(0.0, min(x2, w))
        y2 = max(0.0, min(y2, h))
        bw, bh = x2 - x1, y2 - y1
        if bw <= 0 or bh <= 0:
            skipped_degenerate += 1
            continue

        x_center = min(max((x1 + bw / 2) / w, 0.0), 1.0)
        y_center = min(max((y1 + bh / 2) / h, 0.0), 1.0)
        norm_w = min(max(bw / w, 0.0), 1.0)
        norm_h = min(max(bh / h, 0.0), 1.0)
        cls_id = CLASS_TO_IDX[box["class"]]
        label_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
    yolo_records.append((ann["file"], label_lines))

print(f"Prepared YOLO labels for {len(yolo_records)} images.")
print(f"Skipped {skipped_degenerate} degenerate boxes (zero or negative width/height).")


Prepared YOLO labels for 3087 images.
Skipped 0 degenerate boxes (zero or negative width/height).


### 3. Split into train/val/test and write images + label files

In [6]:
random.shuffle(yolo_records)
n = len(yolo_records)
n_train = int(n * TRAIN_SPLIT)
n_val = int(n * VAL_SPLIT)
split_map = {
    "train": yolo_records[:n_train],
    "val": yolo_records[n_train:n_train + n_val],
    "test": yolo_records[n_train + n_val:],
}

for split_name, records in split_map.items():
    img_split_dir = os.path.join(DETECTION_IMAGES_DIR, split_name)
    lbl_split_dir = os.path.join(DETECTION_LABELS_DIR, split_name)
    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(lbl_split_dir, exist_ok=True)
    for fname, label_lines in records:
        stem = fname.rsplit(".", 1)[0]
        shutil.copy(os.path.join(RAW_IMAGES_DIR, fname), os.path.join(img_split_dir, fname))
        with open(os.path.join(lbl_split_dir, f"{stem}.txt"), "w") as f:
            f.write("\n".join(label_lines))
    print(f"{split_name}: {len(records)} images")


train: 2160 images
val: 463 images
test: 464 images


### 4. Write the data.yaml config for YOLOv8

In [7]:
data_yaml = {
    "path": DETECTION_DIR,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(SELECTED_CLASSES),
    "names": SELECTED_CLASSES,
}
with open(DETECTION_YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"Wrote {DETECTION_YAML_PATH}")
print(open(DETECTION_YAML_PATH).read())


Wrote /content/drive/MyDrive/SmartVisionAI/detection/data.yaml
path: /content/drive/MyDrive/SmartVisionAI/detection
train: images/train
val: images/val
test: images/test
nc: 25
names:
- car
- truck
- bus
- motorcycle
- bicycle
- airplane
- person
- traffic light
- stop sign
- bench
- dog
- cat
- horse
- bird
- cow
- elephant
- bottle
- cup
- bowl
- pizza
- cake
- chair
- couch
- bed
- potted plant



**Next notebook:** `11_Train_YOLO.ipynb` fine-tunes YOLOv8 on this exact dataset and directory structure.